In [5]:
import numpy as np
import matplotlib.pyplot as plt

N_ROWS, N_COLS = 6,8
START,GOAL = (5,0),(0,7)
TRAPS={(1,2),(1,5),(2,2),(2,5),(4,1),(4,3),(4,5)}
ACTIONS=[0,1,2,3] # 0:haut, 1:droite, 2:bas, 3:gauche
DELTAS=[(-1,0),(0,1),(1,0),(0,-1)] # correspond aux actions

class GridWorld:

    def reset(self):
        self.state = START
        self.steps = 0
        return self.state

    def step(self, action):
        dr,dc = DELTAS[action]
        r,c = self.state
        nr=min(max(r+dr,0),N_ROWS-1)
        nc=min(max(c+dc,0),N_COLS-1)
        self.state = (nr,nc)
        self.steps += 1

        if self.state==GOAL:
            return self.state, 20.0, True
        if self.state in TRAPS:
            return self.state, -20.0, True
        if self.steps>=150:
            return self.state, -1.0, True
        return self.state, -1.0, False

    
def state_id(state):
    return state[0]*N_COLS+state[1]

def epsilon_greedy(Q, state, epsilon,rng):
    s=state_id(state)
    if rng.random()<epsilon:
        return int(rng.integers(4))
    best=np.flatnonzero(Q[s]==Q[s].max())
    return int(rng.choice(best))

Verfifications prealables

In [6]:
# 1. Création de l'environnement et vérification du point de départ
env = GridWorld()
assert env.reset() == (5, 0)
print("Reset:", env.reset())

# 2. Test des quatre actions depuis (5, 0)
expected_from_start = {
    0: ((4, 0), -1.0, False),  # haut
    1: ((5, 1), -1.0, False),  # droite
    2: ((5, 0), -1.0, False),  # bas, sur la bordure
    3: ((5, 0), -1.0, False),  # gauche, sur la bordure
}

for action, expected in expected_from_start.items():
    env.reset()
    result = env.step(action)
    assert result == expected, (action, result, expected)
    print("Action", action, "->", result)

# 3. Vérification des déplacements hors grille sur plusieurs bords
env.reset()
assert env.step(2)[0] == (5, 0)  # bas
env.reset()
assert env.step(3)[0] == (5, 0)  # gauche

env.state = (0, 0)
assert env.step(0)[0] == (0, 0)  # haut
env.state = (0, 7)
assert env.step(1)[0] == (0, 7)  # droite
print("Déplacements hors grille: OK")

# 4. Objectif: récompense +20 et fin de l'épisode
env.reset()
env.state = (0, 6)
state, reward, done = env.step(1)
assert state == GOAL
assert reward == 20.0
assert done is True

# Pièges: récompense -20 et fin de l'épisode
for trap in TRAPS:
    env.reset()
    env.state = (trap[0], trap[1] - 1 if trap[1] > 0 else trap[1])
    action = 1 if trap[1] > 0 else 0

    state, reward, done = env.step(action)
    assert state in TRAPS
    assert reward == -20.0
    assert done is True

print("Objectif et pièges: OK")

# 5. Limite de 150 déplacements
env.reset()
done = False
for step_number in range(1, 151):
    state, reward, done = env.step(3)  # reste sur la bordure gauche
    if step_number < 150:
        assert done is False

assert step_number == 150
assert done is True
assert reward == -1.0
print("Limite de 150 déplacements: OK")


Reset: (5, 0)
Action 0 -> ((4, 0), -1.0, False)
Action 1 -> ((5, 1), -1.0, False)
Action 2 -> ((5, 0), -1.0, False)
Action 3 -> ((5, 0), -1.0, False)
Déplacements hors grille: OK
Objectif et pièges: OK
Limite de 150 déplacements: OK


Implementation de Q-Learning

ecrire une fonction train_q_learning(env,episodes,alpha,gamma,epsilon,seed) qui initialise une table Q-Learning de dimension(N_ROWS,N_COLS,4), entraine l'agent et retourne Q ainsi que les recompenses cumulées par episode

In [13]:
def train_q_learning(env, episodes=2000, alpha=0.10, gamma=0.95,
                     epsilon=0.10, seed=0, verbose=False):
    """Entraîne l'agent avec Q-Learning."""
    Q = np.zeros((N_ROWS * N_COLS, 4))
    episode_returns = []
    rng = np.random.default_rng(seed)

    for episode in range(episodes):
        state = env.reset()
        done = False
        total_reward = 0.0

        while not done:
            state_index = state_id(state)
            action = epsilon_greedy(Q, state, epsilon, rng)
            next_state, reward, done = env.step(action)
            next_state_index = state_id(next_state)

            target = reward
            if not done:
                target += gamma * np.max(Q[next_state_index])

            Q[state_index, action] += alpha * (
                target - Q[state_index, action]
            )

            total_reward += reward
            if verbose and episode < 10:
                print((state, action, reward, next_state, done))
            state = next_state

        episode_returns.append(total_reward)

    return Q, np.asarray(episode_returns)


# Étape de validation sur 10 épisodes avec transitions affichées.
env = GridWorld()
Q_debug, rewards_debug = train_q_learning(
    env, episodes=10, alpha=0.10, gamma=0.95,
    epsilon=0.10, seed=0, verbose=True
)
print("Formes après 10 épisodes:", Q_debug.shape, rewards_debug.shape)

# Entraînement complet reproductible.
env = GridWorld()
Q, rewards = train_q_learning(
    env, episodes=2000, alpha=0.10, gamma=0.95,
    epsilon=0.10, seed=0
)
print("Formes finales:", Q.shape, rewards.shape)


((5, 0), 2, -1.0, (5, 0), False)
((5, 0), 1, -1.0, (5, 1), False)
((5, 1), 0, -20.0, (4, 1), True)
((5, 0), 3, -1.0, (5, 0), False)
((5, 0), 0, -1.0, (4, 0), False)
((4, 0), 2, -1.0, (5, 0), False)
((5, 0), 2, -1.0, (5, 0), False)
((5, 0), 3, -1.0, (5, 0), False)
((5, 0), 0, -1.0, (4, 0), False)
((4, 0), 3, -1.0, (4, 0), False)
((4, 0), 1, -20.0, (4, 1), True)
((5, 0), 1, -1.0, (5, 1), False)
((5, 1), 1, -1.0, (5, 2), False)
((5, 2), 1, -1.0, (5, 3), False)
((5, 3), 0, -20.0, (4, 3), True)
((5, 0), 0, -1.0, (4, 0), False)
((4, 0), 0, -1.0, (3, 0), False)
((3, 0), 3, -1.0, (3, 0), False)
((3, 0), 1, -1.0, (3, 1), False)
((3, 1), 1, -1.0, (3, 2), False)
((3, 2), 2, -1.0, (4, 2), False)
((4, 2), 2, -1.0, (5, 2), False)
((5, 2), 2, -1.0, (5, 2), False)
((5, 2), 3, -1.0, (5, 1), False)
((5, 1), 3, -1.0, (5, 0), False)
((5, 0), 1, -1.0, (5, 1), False)
((5, 1), 2, -1.0, (5, 1), False)
((5, 1), 2, -1.0, (5, 1), False)
((5, 1), 1, -1.0, (5, 2), False)
((5, 2), 0, -1.0, (4, 2), False)
((4, 2), 1